In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class CustomMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim//num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.size()
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        Q_h = Q.reshape(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)   #B, num_heads, T, d
        K_h = K.reshape(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)   #B, num_heads, T, d
        V_h = V.reshape(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)   #B, num_heads, T, d
        scores = (Q_h@K_h.permute(0,1,3,2))/math.sqrt(self.head_dim)    #B, num_heads, T, T
        attention_weights = F.softmax(scores, dim=-1)   #B, H, T, T
        A = attention_weights@V_h   #B, H, T, D
        A = A.permute(0,2,1,3).contiguous().view(B, T, self.num_heads*self.head_dim)
        self.out_proj(A)
        return A

# --- ТЕСТИРОВАНИЕ ---
# Если код написан верно, размерность на выходе совпадет с размерностью на входе
torch.manual_seed(42)
B, T, C = 2, 8, 512 # 2 текста, 8 токенов в каждом, 512 размерность эмбеддинга
x = torch.randn(B, T, C)

mha = CustomMultiHeadAttention(embed_dim=C, num_heads=8)
out = mha(x)

print(f"Входной shape: {x.shape}")
print(f"Выходной shape: {out.shape}")
assert x.shape == out.shape, "Ошибка: размерности не совпадают!"
print("Успех! Attention вычислен верно.")

Входной shape: torch.Size([2, 8, 512])
Выходной shape: torch.Size([2, 8, 512])
Успех! Attention вычислен верно.


In [5]:
from typing import Any


class RMSNorm(nn.Module):
    def __init__(self, dim, eps = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self,x):
        # ТВОЯ ЗАДАЧА №1: Реализовать RMSNorm
        # 1. Возведи x в квадрат
        # 2. Найди среднее значение по последней оси (dim=-1, keepdim=True)
        # 3. Прибавь self.eps (для стабильности, чтобы не делить на ноль)
        # 4. Извлеки квадратный корень (это и есть RMS - Root Mean Square)
        # 5. Подели x на RMS и умножь на self.weight
        # ...
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        y = x/rms * self.weight
        return y


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self,x):
        # ТВОЯ ЗАДАЧА №2: Реализовать SwiGLU
        # Формула: ( SiLU(x * W1) * (x * W2) ) * W3
        # Подсказка: функция SiLU в PyTorch это F.silu()
        # Оператор * здесь - это поэлементное умножение (Hadamard product)
        # ...
        x = self.w1(x) * F.silu(self.w2(x))
        x = self.w3(x)
        return x

class LlamaTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = CustomMultiHeadAttention(embed_dim=dim, num_heads=num_heads)
        self.feed_forward = SwiGLU(dim=dim, hidden_dim=hidden_dim)

        self.attention_norm = RMSNorm(dim=dim)
        self.ffn_norm = RMSNorm(dim=dim)

    def forward(self, x):
        # ТВОЯ ЗАДАЧА №3: Собрать Pre-Norm архитектуру с Residual связями
        # 1. Сохрани x (residual)
        # 2. Пропусти x через attention_norm
        # 3. Пропусти результат через attention
        # 4. Прибавь residual к результату (это новый x)

        # 5. Сохрани новый x (новый residual)
        # 6. Пропусти x через ffn_norm
        # 7. Пропусти результат через feed_forward
        # 8. Прибавь residual к результату

        # Верни итоговый x
        # ...
        x = x + self.attention(self.attention_norm(x))
        x = x + self.feed_forward(self.ffn_norm(x))
        return x
    # TESTING
torch.manual_seed(42)
B, T, C = 2, 8, 512
hidden_dim = 2048
x = torch.randn(B, T, C)

block = LlamaTransformerBlock(dim=C, num_heads=8, hidden_dim=hidden_dim)
out = block(x)
print(f"Входной shape: {x.shape}")
print(f"Выходной shape: {out.shape}")
assert x.shape == out.shape, "Ошибка: размерности не совпадают!"
print("Успех! Блок Трансформера собран верно.")

Входной shape: torch.Size([2, 8, 512])
Выходной shape: torch.Size([2, 8, 512])
Успех! Блок Трансформера собран верно.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- ВСТАВЬ СЮДА СВОЙ КОД ИЗ ДНЯ 1 И ДНЯ 2 ---
# class CustomMultiHeadAttention(nn.Module): ...
# class RMSNorm(nn.Module): ...
# class SwiGLU(nn.Module): ...
# class LlamaTransformerBlock(nn.Module): ...
# ---------------------------------------------

def precompute_rope_angles(dim, seq_len, theta=10000.0):
    """
    Магия RoPE (Rotary Positional Embedding).
    Мы заранее вычисляем углы поворота для каждой позиции токена.
    """
    # 1. Вычисляем частоты для каждой пары измерений
    # Используем torch.arange для создания последовательности [0, 2, 4... dim]
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))

    # 2. Создаем позиции токенов [0, 1, 2... seq_len-1]
    t = torch.arange(seq_len, dtype=torch.float32)

    # 3. Внешнее произведение (каждая позиция умножается на каждую частоту)
    # Используем torch.outer!
    freqs_outer = torch.outer(t, freqs)

    # 4. Превращаем углы в комплексные числа (cos + i*sin) для быстрого вращения
    # torch.polar(абсолютное значение=1.0, угол=freqs_outer)
    complex_angles = torch.polar(torch.ones_like(freqs_outer), freqs_outer)
    return complex_angles

class MiniLlama(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers):
        super().__init__()
        # ТВОЯ ЗАДАЧА №1: Создать Embedding слой (используй nn.Embedding)
        self.tok_embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim) # Напиши код

        # ТВОЯ ЗАДАЧА №2: Создать список блоков Трансформера (используй nn.ModuleList)
        # Нам нужно создать num_layers штук LlamaTransformerBlock
        self.layers = nn.ModuleList([
            LlamaTransformerBlock(dim=embed_dim, num_heads=num_heads, hidden_dim=hidden_dim) for _ in range(num_layers)
            # Напиши код генерации блоков
        ])

        # ТВОЯ ЗАДАЧА №3: Финальный RMSNorm и Линейный слой (lm_head)
        # Линейный слой должен перевести из embed_dim обратно в vocab_size
        self.norm = RMSNorm(embed_dim) # Напиши код
        self.lm_head = nn.Linear(embed_dim, vocab_size) # Напиши код

    def forward(self, tokens):
        B, T = tokens.size()

        # 1. Получаем эмбеддинги токенов
        x = self.tok_embeddings(tokens) # [B, T, C]

        # (В реальной Llama здесь применяется RoPE к Query и Key внутри Attention,
        # но для простоты архитектуры сегодня мы пропустим интеграцию RoPE внутрь твоего
        # CustomMultiHeadAttention, чтобы не сломать вчерашний код. Оставим RoPE на десерт).

        # 2. Пропускаем через все слои Трансформера
        for layer in self.layers:
            x = layer(x)

        # 3. Финальная нормализация
        x = self.norm(x)

        # 4. Получаем логиты (вероятности) для словаря
        logits = self.lm_head(x) # [B, T, Vocab_Size]

        return logits

# --- ТЕСТИРОВАНИЕ ---
torch.manual_seed(42)
VOCAB_SIZE = 1000  # Допустим, наш словарь - 1000 слов
B, T = 2, 8        # 2 текста, по 8 токенов
# Генерируем фейковые ID токенов (числа от 0 до 999)
fake_tokens = torch.randint(0, VOCAB_SIZE, (B, T))

model = MiniLlama(
    vocab_size=VOCAB_SIZE,
    embed_dim=256,
    num_heads=8,
    hidden_dim=1024,
    num_layers=4 # 4 слоя Трансформера!
)

# Прогоняем данные через всю модель
logits = model(fake_tokens)

print(f"Входной shape (токены): {fake_tokens.shape}")
print(f"Выходной shape (логиты): {logits.shape}")
assert logits.shape == (B, T, VOCAB_SIZE), "Ошибка: размерность выхода неверная!"
print("Успех! Твоя первая LLM собрана и готова к обучению.")